# Ejercicio 5 — Actualización de registros en datasets

**Seminario de Lenguajes Opción Python — TI 2026**

En esta sección se aplican las funciones de actualización del módulo `actualizacion.py`.  
Toda operación genera un nuevo archivo en `processed_datasets/`; los originales nunca se modifican.

---

In [ ]:
import sys
sys.path.insert(0, '../src')

import actualizacion

## Configuración

In [ ]:
# ── Ajustar según el dataset a utilizar ──────────────────────────────────────
RUTA_DATASET   = '../raw_datasets/occurrence.txt'
RUTA_SALIDA    = '../processed_datasets/occurrence_actualizado.txt'
SEPARADOR      = '\t'
ENCODING       = 'utf-8'
# ─────────────────────────────────────────────────────────────────────────────

---
## 5.A — Búsqueda de registros por múltiples columnas

La función recibe un diccionario de filtros `{columna: valor}` y retorna todos los registros que cumplan **todas** las condiciones.

In [ ]:
# Ejemplo 1: buscar por occurrenceID
resultados = actualizacion.buscar_registros(
    RUTA_DATASET,
    filtros={'occurrenceID': '1234'},
    separador=SEPARADOR,
    encoding=ENCODING,
)
print(f"Registros encontrados (por ID): {len(resultados)}")
for r in resultados[:3]:
    print(r)

In [ ]:
# Ejemplo 2: buscar por scientificName y country
resultados = actualizacion.buscar_registros(
    RUTA_DATASET,
    filtros={'scientificName': 'Elaenia chilensis', 'country': 'Argentina'},
    separador=SEPARADOR,
    encoding=ENCODING,
)
print(f"Registros encontrados (nombre + país): {len(resultados)}")
for r in resultados[:3]:
    print(r)

In [ ]:
# Ejemplo 3: buscar solo por country
resultados = actualizacion.buscar_registros(
    RUTA_DATASET,
    filtros={'country': 'Argentina'},
    separador=SEPARADOR,
    encoding=ENCODING,
)
print(f"Registros encontrados (país = Argentina): {len(resultados)}")

---
## 5.B — Actualizar un campo de un registro

Recibe el ID del registro, el nombre de la columna y el nuevo valor.  
Genera un nuevo archivo en `processed_datasets/`.

In [ ]:
resultado = actualizacion.actualizar_campo(
    ruta_archivo=RUTA_DATASET,
    ruta_salida=RUTA_SALIDA,
    id_registro='1234',           # reemplazar con un ID real del dataset
    nombre_columna='stateProvince',
    nuevo_valor='Buenos Aires',
    separador=SEPARADOR,
    encoding=ENCODING,
)

print("Resultado:")
print(f"  Éxito              : {resultado['exito']}")
print(f"  Registros modificados: {resultado['registros_modificados']}")
if resultado['errores']:
    print(f"  Errores            : {resultado['errores']}")

---
## 5.C — Actualizar múltiples campos en una sola operación

Recibe un diccionario `{columna: nuevo_valor}` con todos los campos a modificar.

In [ ]:
cambios = {
    'country': 'Argentina',
    'stateProvince': 'Buenos Aires',
}

resultado = actualizacion.actualizar_campos(
    ruta_archivo=RUTA_DATASET,
    ruta_salida=RUTA_SALIDA,
    id_registro='1234',           # reemplazar con un ID real del dataset
    cambios=cambios,
    separador=SEPARADOR,
    encoding=ENCODING,
)

print("Resultado:")
print(f"  Éxito              : {resultado['exito']}")
print(f"  Registros modificados: {resultado['registros_modificados']}")
if resultado['errores']:
    print(f"  Errores: {resultado['errores']}")

---
## 5.D — Actualizar con validación previa

Antes de aplicar los cambios se validan los nuevos valores reutilizando  
las funciones del módulo `validacion.py`.  
Si alguna validación falla el registro **no se modifica** y se reportan los errores.

In [ ]:
# Caso exitoso: valores correctos
cambios_validos = {
    'country': 'Argentina',
    'countryCode': 'AR',
    'decimalLatitude': '-34.6',
    'decimalLongitude': '-58.4',
}

resultado = actualizacion.actualizar_campos_con_validacion(
    ruta_archivo=RUTA_DATASET,
    ruta_salida=RUTA_SALIDA,
    id_registro='1234',
    cambios=cambios_validos,
    separador=SEPARADOR,
    encoding=ENCODING,
)

print("--- Caso con valores válidos ---")
print(f"  Éxito              : {resultado['exito']}")
print(f"  Registros modificados: {resultado['registros_modificados']}")
print(f"  Errores            : {resultado['errores']}")

In [ ]:
# Caso fallido: valores incorrectos (la modificación NO se aplica)
cambios_invalidos = {
    'countryCode': 'XX',          # código inexistente
    'decimalLatitude': '999',     # fuera de rango
}

resultado = actualizacion.actualizar_campos_con_validacion(
    ruta_archivo=RUTA_DATASET,
    ruta_salida=RUTA_SALIDA,
    id_registro='1234',
    cambios=cambios_invalidos,
    separador=SEPARADOR,
    encoding=ENCODING,
)

print("--- Caso con valores inválidos ---")
print(f"  Éxito              : {resultado['exito']}")
print(f"  Registros modificados: {resultado['registros_modificados']}")
print(f"  Errores:")
for e in resultado['errores']:
    print(f"    - {e}")